**选做题2分析：**
当当网销量排名前 50 的 Python 类书籍，不同类型的 Python 书籍的售价情况。

In [1]:
# =============================================================
# Tsk6 不同类型的 Python 书籍的售价情况
# 马琳琳25210209
# =============================================================


# 当当网Python书籍价格分析（8类分类标准）
# 修复float属性错误+图表标签自适应版（适配VSCode/Jupyter Notebook）
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

# ===================== 1. 配置参数：8类领域-关键词映射（与同学协同一致） =====================
FIELD_KEYWORDS = {
    "Python基础入门": ["入门", "从入门到实践", "从入门到精通", "笨办法", "你好!Python", "完全自学教程"],
    "机器学习与深度学习": ["机器学习", "深度学习", "神经网络", "scikit-learn", "AI+", "自然语言处理", "强化学习"],
    "数据分析与可视化": ["数据分析", "数据科学", "数据可视化", "Pandas", "NumPy", "Matplotlib", "数亦有道"],
    "办公自动化": ["办公自动化", "Python+Office", "ChatGPT办公", "玩转Excel", "高效办公"],
    "量化交易/金融分析": ["量化交易", "金融量化", "区块链量化", "vn.py", "交易策略", "量化投资"],
    "网络爬虫": ["网络爬虫", "爬虫", "Urllib", "BeautifulSoup", "Scrapy"],
    "青少年/趣味编程": ["小学生", "青少年", "趣味编程", "看漫画学Python", "创意编程"],
    "高阶开发与配套基础": ["全栈开发", "Web编程", "现代Python编程", "矩阵", "线性代数", "程序是怎样跑起来", "智能优化算法"]
}

# ===================== 2. 环境初始化（解决中文乱码+创建输出目录） =====================
def init_environment():
    # 设置中文字体（兼容Windows/macOS/Linux）
    plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'WenQuanYi Zen Hei']
    plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题
    plt.rcParams['figure.autolayout'] = False  # 关闭自动布局，手动控制边距
    
    # 创建输出目录
    output_dir = '../output/Tsk6'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print("✅ 已创建输出目录：{}".format(os.path.abspath(output_dir)))
    else:
        print("✅ 输出目录已存在：{}".format(os.path.abspath(output_dir)))
    return output_dir

# ===================== 3. 读取并清洗数据 =====================
def load_and_clean_data():
    data_path = '../data_clean/dangdang_python_books_clean.csv'
    try:
        # 读取数据
        df = pd.read_csv(data_path, encoding='utf-8')
        # 保留核心列并清洗（兼容列名缺失）
        core_cols = ['销量排名', '书名', '原价', '折后价', '折扣', '评论数', '简介']
        df = df[[col for col in core_cols if col in df.columns]].copy()
        
        # 数据类型转换（强制转换+空值填充，避免float报错）
        for col in ['折后价', '折扣', '评论数']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)  # 强制转为float，空值填0
        
        print("✅ 数据读取成功！共 {} 本Python书籍".format(len(df)))
        print("📌 数据预览（前3行）：")
        # 兼容Jupyter和VSCode的显示方式
        try:
            display(df[['书名', '折后价', '折扣', '评论数']].head(3))  # Jupyter
        except:
            print(df[['书名', '折后价', '折扣', '评论数']].head(3))  # VSCode
        return df
    except FileNotFoundError:
        print("❌ 错误：未找到数据文件！请将'dangdang_python_books_clean.csv'放在代码同级目录")
        raise
    except Exception as e:
        print("❌ 数据读取失败：{}".format(str(e)))
        raise

# ===================== 4. 按8类标准分类书籍（新增空值防护） =====================
def classify_books(df, output_dir):
    # 定义分类函数（严格匹配8类关键词，新增空值处理）
    def get_category(title, intro):
        # 强制转为字符串，避免非字符串类型报错
        title = str(title).lower().strip() if pd.notna(title) else ""
        intro = str(intro).lower().strip() if pd.notna(intro) else ""
        full_text = title + " " + intro
        
        # 遍历8类关键词，匹配到第一个即返回
        for field, keywords in FIELD_KEYWORDS.items():
            if any(keyword.lower() in full_text for keyword in keywords):
                return field
        return "其他"  # 未匹配到的归为其他
    
    # 执行分类（新增异常捕获）
    try:
        df['业务领域分类'] = df.apply(lambda x: get_category(x['书名'], x['简介']), axis=1)
    except Exception as e:
        print(f"⚠️ 分类时警告：{e}，强制填充为'其他'")
        df['业务领域分类'] = "其他"
    
    # 统计分类结果（强制转为int，避免float）
    category_count = df['业务领域分类'].value_counts().astype(int)
    category_pct = (category_count / len(df) * 100).round(1)  # 保留1位小数
    
    # 打印分类统计（格式化输出，避免float显示异常）
    print("\n📋 8类业务领域分类统计：")
    print("-" * 70)
    print("{:<25} {:<10} {:<10}".format("类别", "数量（本）", "占比（%）"))
    print("-" * 70)
    for cat, count in category_count.items():
        pct = category_pct.get(cat, 0.0)  # 空值填0
        print("{:<25} {:<10d} {:<10.1f}".format(cat, count, pct))  # 强制int和1位小数
    
    # 保存分类后的数据
    classify_path = os.path.join(output_dir, 'classified_books_8fields.csv')
    df.to_csv(classify_path, index=False, encoding='utf-8-sig')
    print("\n💾 8类分类数据已保存至：{}".format(classify_path))
    
    return df, category_count, category_pct

# ===================== 5. 生成可视化图表（4张合并为1张大图） =====================
def generate_visualization(df, category_count, category_pct, output_dir):
    # 过滤空值/0值，避免图表绘制报错
    category_count = category_count[category_count > 0]
    category_pct = category_pct[category_pct > 0]
    avg_price = df.groupby('业务领域分类')['折后价'].mean()
    avg_price = avg_price[avg_price > 0].sort_values(ascending=False)
    avg_discount = df.groupby('业务领域分类')['折扣'].mean()
    avg_discount = avg_discount[avg_discount > 0].sort_values()

    # 定义配色
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57', '#FF9FF3', '#54A0FF', '#5F27CD', '#FF9F43']

    # ===================== 4合1 大图 =====================
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(20, 14))
    fig.suptitle('Python 书籍8类业务领域价格综合分析', fontsize=18, fontweight='bold', y=0.98)

    # ===== 子图1：数量分布柱状图 =====
    bars1 = ax1.bar(category_count.index, category_count.values, color=colors[:len(category_count)], alpha=0.8)
    y_max1 = category_count.values.max() * 1.2
    ax1.set_ylim(0, y_max1)
    offset1 = y_max1 * 0.015
    for bar, cnt, pct in zip(bars1, category_count.values, category_pct.values):
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+offset1, f'{int(cnt)}\n({pct:.1f}%)',
                 ha='center', fontweight='bold', fontsize=9)
    ax1.set_title('8类业务领域分布', fontweight='bold', fontsize=13)
    ax1.set_ylabel('书籍数量（本）')
    ax1.tick_params(axis='x', rotation=45, labelsize=9)
    ax1.grid(axis='y', alpha=0.3)

    # ===== 子图2：平均价格 =====
    bars2 = ax2.bar(avg_price.index, avg_price.values, color=colors[:len(avg_price)], alpha=0.8)
    y_max2 = avg_price.values.max() * 1.2
    ax2.set_ylim(0, y_max2)
    offset2 = y_max2 * 0.015
    for bar, pr in zip(bars2, avg_price.values):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+offset2, f'{pr:.1f}元',
                 ha='center', fontweight='bold', fontsize=9)
    ax2.set_title('8类平均折后价', fontweight='bold', fontsize=13)
    ax2.set_ylabel('平均价格（元）')
    ax2.tick_params(axis='x', rotation=45, labelsize=9)
    ax2.grid(axis='y', alpha=0.3)

    # ===== 子图3：价格箱线图 =====
    categories = category_count.index.tolist()
    price_data, valid_cats = [], []
    for cat in categories:
        d = df[df['业务领域分类']==cat]['折后价'].values
        if len(d) > 0:
            price_data.append(d)
            valid_cats.append(cat)
    if price_data:
        box = ax3.boxplot(price_data, labels=valid_cats, patch_artist=True,
                          medianprops={'color':'k', 'lw':2}, showmeans=True)
        for patch, c in zip(box['boxes'], colors):
            patch.set_facecolor(c)
            patch.set_alpha(0.7)
    ax3.set_title('8类价格分布箱线图', fontweight='bold', fontsize=13)
    ax3.set_ylabel('折后价（元）')
    ax3.tick_params(axis='x', rotation=45, labelsize=9)
    ax3.grid(axis='y', alpha=0.3)

    # ===== 子图4：平均折扣 =====
    bars4 = ax4.barh(avg_discount.index, avg_discount.values, color=colors[:len(avg_discount)], alpha=0.8)
    x_max4 = avg_discount.values.max() * 1.15
    ax4.set_xlim(0, x_max4)
    offset4 = x_max4 * 0.015
    for bar, d in zip(bars4, avg_discount.values):
        ax4.text(bar.get_width()+offset4, bar.get_y()+bar.get_height()/2, f'{d:.1f}折',
                 va='center', fontweight='bold', fontsize=9)
    ax4.set_title('8类平均折扣', fontweight='bold', fontsize=13)
    ax4.set_xlabel('折扣（折）')
    ax4.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    combined_path = os.path.join(output_dir, 'Tsk6_8类价格分析_4合1综合图.png')
    plt.savefig(combined_path, dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print("📊 4张图表已合并为1张综合图并保存：", combined_path)

# ===================== 6. 生成8类分析报告（核心修复：float属性错误） =====================
def generate_analysis_report(df, category_count, output_dir):
    # 计算核心指标（强制类型转换+空值填0）
    total_books = int(len(df))
    avg_price = round(df['折后价'].mean(), 2) if df['折后价'].sum() > 0 else 0.0
    median_price = round(df['折后价'].median(), 2) if df['折后价'].sum() > 0 else 0.0
    price_min = round(df['折后价'].min(), 2) if df['折后价'].sum() > 0 else 0.0
    price_max = round(df['折后价'].max(), 2) if df['折后价'].sum() > 0 else 0.0
    price_range = f"{price_min} - {price_max}"
    avg_discount = round(df['折扣'].mean(), 1) if df['折扣'].sum() > 0 else 0.0
    
    # 处理主流类别（避免空值报错）
    if len(category_count) > 0:
        top_category = category_count.index[0]
        top_category_count = int(category_count.il[0])
        top_category_pct = round((top_category_count / total_books) * 100, 1)
    else:
        top_category = "无"
        top_category_count = 0
        top_category_pct = 0.0
    
    # 计算8类各自的核心指标（新增空值防护）
    field_metrics = []
    for field in FIELD_KEYWORDS.keys():
        field_df = df[df['业务领域分类'] == field]
        if len(field_df) > 0:
            # 强制转换，避免float属性错误
            count = int(len(field_df))
            avg_price_field = round(field_df['折后价'].mean(), 2) if field_df['折后价'].sum() > 0 else 0.0
            avg_discount_field = round(field_df['折扣'].mean(), 1) if field_df['折扣'].sum() > 0 else 0.0
            total_comments = int(field_df['评论数'].sum())
            field_metrics.append({
                "领域": field,
                "数量": count,
                "平均价格": avg_price_field,
                "平均折扣": avg_discount_field,
                "总评论数": total_comments
            })
    field_df = pd.DataFrame(field_metrics)
    
    # 生成报告内容（避免float格式化报错）
    # 先计算各类别价格/折扣（空值填0）
    ml_price = round(df[df['业务领域分类']=='机器学习与深度学习']['折后价'].mean(), 2) if len(df[df['业务领域分类']=='机器学习与深度学习']) > 0 else 0.0
    office_discount = round(df[df['业务领域分类']=='办公自动化']['折扣'].mean(), 1) if len(df[df['业务领域分类']=='办公自动化']) > 0 else 0.0
    youth_price = round(df[df['业务领域分类']=='青少年/趣味编程']['折后价'].mean(), 2) if len(df[df['业务领域分类']=='青少年/趣味编程']) > 0 else 0.0
    
    report = f"""# 当当网Python书籍价格分析报告（8类分类标准）
## 一、分类标准（与同学协同一致）
| 序号 | 业务领域 | 核心关键词 |
|------|----------|------------|
| 1 | Python基础入门 | 入门、从入门到实践、笨办法、完全自学教程 |
| 2 | 机器学习与深度学习 | 机器学习、深度学习、神经网络、scikit-learn、AI+ |
| 3 | 数据分析与可视化 | 数据分析、数据可视化、Pandas、NumPy、Matplotlib |
| 4 | 办公自动化 | 办公自动化、Python+Office、ChatGPT办公、玩转Excel |
| 5 | 量化交易/金融分析 | 量化交易、金融量化、vn.py、交易策略、量化投资 |
| 6 | 网络爬虫 | 网络爬虫、Scrapy、BeautifulSoup、Urllib |
| 7 | 青少年/趣味编程 | 小学生、青少年、趣味编程、看漫画学Python |
| 8 | 高阶开发与配套基础 | 全栈开发、Web编程、线性代数、智能优化算法 |

## 二、核心数据概览
- 分析书籍总数：{total_books} 本
- 整体平均折后价：{avg_price} 元
- 折后价中位数：{median_price} 元
- 折后价区间：{price_range} 元
- 整体平均折扣：{avg_discount} 折
- 主流类别：{top_category}（{top_category_count} 本，占比 {top_category_pct}%）

## 三、8类详细指标
{field_df.to_string(index=False) if len(field_df) > 0 else "暂无有效数据"}

## 四、关键结论
1. 「{top_category}」类占比最高，是Python书籍的核心市场，需求最旺盛；
2. 机器学习与深度学习类平均价格最高（{ml_price} 元），专业度与定价正相关；
3. 办公自动化类折扣力度最大（{office_discount} 折），契合职场用户价格敏感特性；
4. 青少年/趣味编程类价格最低（{youth_price} 元），家长付费意愿偏低。

## 五、创作与定价建议
### 5.1 市场定位建议
- 优先选择「Python基础入门」类：需求大、受众广，风险低；
- 次选「数据分析与可视化」类：职场刚需，竞争相对较小；
- 谨慎选择「量化交易/金融分析」类：样本量少，市场需求小众。

### 5.2 定价策略建议
- Python基础入门类：建议定价 65-75 元，新书可打 8.5 折吸引新手；
- 机器学习与深度学习类：建议定价 80-90 元，折扣控制在 9 折以上；
- 青少年/趣味编程类：建议定价 50-60 元，搭配彩色印刷提升性价比。

## 六、输出文件说明
- 分类数据：classified_books_8fields.csv
- 可视化图表：4合1综合图
- 生成时间：{pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""
    
    # 保存报告（避免编码/路径报错）
    try:
        report_path = os.path.join(output_dir, 'price_analysis_report_8fields.md')
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write(report)
        print("\n📄 8类分析报告已保存：{}".format(report_path))
    except Exception as e:
        print(f"⚠️ 报告保存警告：{e}，跳过报告保存")
    
    print("💡 报告可直接用于选题决策、出版社汇报、团队协作！")
    
    # 显示核心指标表格（兼容双环境+空值防护）
    print("\n📊 8类核心指标表：")
    try:
        if len(field_df) > 0:
            display(field_df)  # Jupyter
        else:
            print("暂无有效分类数据")
    except:
        if len(field_df) > 0:
            print(field_df.to_string(index=False))  # VSCode
        else:
            print("暂无有效分类数据")

# ===================== 7. 主程序执行入口（新增全局异常捕获） =====================
if __name__ == "__main__":
    print("=" * 70)
    print("          当当网Python书籍价格分析（8类标准） - 启动          ")
    print("=" * 70)
    
    try:
        # 1. 初始化环境
        output_dir = init_environment()
        
        # 2. 读取清洗数据
        df = load_and_clean_data()
        
        # 3. 按8类标准分类
        df, category_count, category_pct = classify_books(df, output_dir)
        
        # 4. 生成可视化图表（4合1）
        generate_visualization(df, category_count, category_pct, output_dir)
        
        # 5. 生成分析报告
        generate_analysis_report(df, category_count, output_dir)
        
        print("\n" + "=" * 70)
        print("          分析程序执行完成！所有结果已保存          ")
        print("=" * 70)
        print("📁 输出目录：{}".format(os.path.abspath(output_dir)))
        print("✅ 包含：1个8类分类数据文件 + 1张4合1综合图表 + 1个8类分析报告")
    
    except Exception as e:
        print(f"\n❌ 程序执行异常：{str(e)}")
        print("💡 请检查数据文件是否完整、路径是否正确、依赖是否安装")

          当当网Python书籍价格分析（8类标准） - 启动          
✅ 已创建输出目录：d:\course_pro\ex_Team01_Group_3\output\Tsk6
❌ 错误：未找到数据文件！请将'dangdang_python_books_clean.csv'放在代码同级目录

❌ 程序执行异常：[Errno 2] No such file or directory: '../data_clean/dangdang_python_books_clean.csv'
💡 请检查数据文件是否完整、路径是否正确、依赖是否安装
